# Pritam — Day 4 Session Summary
**Sprint:** Dark Store + Integrated Logistics | **Date:** April 4, 2026  
**Branch:** `pritam_temp_apr1` | **Role:** Lead Coder + Integration Architect

---

**Day 4 scope:**  
1. Return Probability Classifier (XGBoost) → `data/master_df_v3.parquet`  
2. Reverse VRP (CVRPTW pickup routing) → `outputs/reverse_routes.json`  
3. Joint Optimizer Z function → `outputs/joint_optimizer_result.json`

Load [`docs/pritam_sess_summ.md`](../../docs/pritam_sess_summ.md) for Pritam-scoped context.

---
## 1. Step 1 — Return Probability Classifier

### What `return_classifier.run_full_pipeline()` does
```
1. load data            — master_df_v2.parquet (19,207 rows, 2.94% return rate)
2. build_features()     — encode categoricals (LabelEncoder), median-impute numerics
3. train()              — XGBClassifier + Platt calibration (cv=3), scale_pos_weight=33
4. evaluate()           — ROC-AUC, PR-AUC, Brier score @ threshold=0.30
5. add_return_prob()    — append return_prob (float32) + return_flag (0/1)
6. save outputs         — master_df_v3.parquet, return_clf_v1.pkl, metrics.json
```

### Features used

| Type | Columns |
|------|---------|
| Numeric | `review_score`, `freight_value`, `order_value`, `product_weight_g` |
| Categorical (label-encoded) | `product_category_name_english`, `payment_type`, `seller_state` |

### Evaluation results

| Metric | Value | Target |
|--------|-------|--------|
| ROC-AUC | **0.8969** | ≥ 0.70 ✅ |
| PR-AUC | 0.4716 | — |
| Brier score | 0.0213 | — |
| Precision @ 0.30 | 0.5556 | — |
| Recall @ 0.30 | 0.3540 | — |
| F1 @ 0.30 | 0.4324 | — |
| Orders flagged | **593 / 19,207 (3.1%)** | — |

ROC-AUC 0.897 is strong for 2.9% class imbalance. Platt calibration keeps probabilities well-calibrated.

In [ ]:
import sys
sys.path.insert(0, "../..")

import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

metrics = json.load(open("../../outputs/return_classifier_metrics.json"))
v3 = pd.read_parquet("../../data/master_df_v3.parquet")

print("=== Return Classifier Metrics ===")
print(f"ROC-AUC   : {metrics['roc_auc']}  {'✓' if metrics['roc_auc'] >= 0.70 else '✗'}")
print(f"PR-AUC    : {metrics['pr_auc']}")
print(f"Brier     : {metrics['brier_score']}")
thr = metrics["threshold_0.3"]
print(f"\n@ threshold 0.30:")
print(f"  Precision : {thr['precision']}")
print(f"  Recall    : {thr['recall']}")
print(f"  F1        : {thr['f1']}")
print(f"  Flagged   : {thr['flagged_pct']}% of orders")
print()
print(f"master_df_v3 shape : {v3.shape}")
print(f"return_flag=1 count: {v3['return_flag'].sum():,}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(v3["return_prob"], bins=50, color="steelblue", edgecolor="white")
ax.axvline(0.30, color="red", linestyle="--", label="threshold=0.30")
ax.set_xlabel("return_prob"); ax.set_ylabel("Count")
ax.set_title("Distribution of Return Probabilities (19,207 orders)")
ax.legend()
plt.tight_layout()
plt.savefig("../../outputs/day4_return_prob_hist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/day4_return_prob_hist.png")

---
## 2. Step 2 — Reverse VRP (CVRPTW Pickup Routing)

### What `reverse_vrp.run_full_pipeline()` does
```
1. filter                      — master_df_v3 → 593 return_flag=1 orders
2. build_reverse_vrp_nodes()   — 11 zones: 1 dark store depot + pickup nodes per zone
3. solve_reverse_cvrptw()      — OR-Tools CVRPTW, PATH_CHEAPEST_ARC + SIMULATED_ANNEALING
4. save_routes()               — reverse_routes.csv/json, reverse_kpi_summary.csv
```

### Zone-level results

| Zone | Pickups | Vehicles | Distance (km) | Cost (R$) |
|------|---------|----------|---------------|-----------|
| 0 | 51 | 1 | 83.0 | 174 |
| 1 | 75 | 2 | 81.5 | 222 |
| 2 | 57 | 2 | 107.0 | 261 |
| 3 | 34 | 2 | 59.3 | 189 |
| 4 | 46 | 1 | 90.0 | 185 |
| 5 | 42 | 1 | 93.7 | 191 |
| 6 | 45 | 1 | 80.8 | 171 |
| 7 | 53 | 1 | 75.9 | 164 |
| 8 | 45 | 2 | 100.0 | 250 |
| 9 | 57 | 1 | 87.8 | 182 |
| 10 | 71 | 1 | 87.4 | 181 |
| **Total** | **576** | **15** | **946.4** | **R$2,170** |

All 11/11 zones solved. Forward R$2,704 + Reverse R$2,170 = **R$4,874 combined logistics cost**.

In [ ]:
rev_kpi = pd.read_csv("../../outputs/reverse_kpi_summary.csv")
fwd_kpi = pd.read_csv("../../outputs/forward_kpi_summary.csv")

print("=== Reverse VRP KPI Summary ===")
print(rev_kpi[["zone_id","n_pickups","n_vehicles_used","total_dist_km","routing_cost_R$"]].to_string(index=False))
print()
print(f"Total pickups   : {rev_kpi['n_pickups'].sum()}")
print(f"Total distance  : {rev_kpi['total_dist_km'].sum():.1f} km")
print(f"Total cost      : R${rev_kpi['routing_cost_R$'].sum():.0f}")
print(f"Total vehicles  : {rev_kpi['n_vehicles_used'].sum()}")

width = 0.35
x = fwd_kpi["zone_id"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(x - width/2, fwd_kpi["total_dist_km"], width, label="Forward", color="steelblue")
axes[0].bar(x + width/2, rev_kpi["total_dist_km"], width, label="Reverse", color="coral")
axes[0].set_xlabel("Zone"); axes[0].set_ylabel("km")
axes[0].set_title("Forward vs Reverse Distance per Zone")
axes[0].legend(); axes[0].set_xticks(x)
axes[1].bar(x - width/2, fwd_kpi["routing_cost_R$"], width, label="Forward", color="steelblue")
axes[1].bar(x + width/2, rev_kpi["routing_cost_R$"], width, label="Reverse", color="coral")
axes[1].set_xlabel("Zone"); axes[1].set_ylabel("R$")
axes[1].set_title("Forward vs Reverse Cost per Zone")
axes[1].legend(); axes[1].set_xticks(x)
plt.tight_layout()
plt.savefig("../../outputs/day4_fwd_vs_rev_kpi.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/day4_fwd_vs_rev_kpi.png")

---
## 3. Step 3 — Joint Optimizer Z Function

### Objective

$$Z = \alpha \cdot C_{fwd} + \beta \cdot C_{rev} + \gamma \cdot T_{pen} + \delta \cdot N_{veh}$$

| Symbol | Meaning | Day 4 value |
|--------|---------|-------------|
| $C_{fwd}$ | Forward route distance (km, active vehicles) | 44.9 km |
| $C_{rev}$ | Reverse route distance (km, active vehicles) | 169.6 km |
| $T_{pen}$ | Expected return penalty (sum of return_prob for flagged) | 287.1 |
| $N_{veh}$ | Active vehicles (forward + reverse) | 3 |
| **Z** | Weighted total | **54.38** |

Weights: α=β=γ=δ=**0.25** (equal — to be tuned on Day 6).

### What the MILP decides
Routes are fixed (OR-Tools). PuLP/CBC selects which vehicles to activate.  
8 binary variables, solved Optimal in 0.04s.

In [ ]:
joint = json.load(open("../../outputs/joint_optimizer_result.json"))

print("=== Joint Optimizer Result ===")
print(f"  Status  : {joint['status']}")
print(f"  Z       : {joint['Z']}")
print(f"  C_fwd   : {joint['C_fwd']} km  (active forward vehicles)")
print(f"  C_rev   : {joint['C_rev']} km  (active reverse vehicles)")
print(f"  T_pen   : {joint['T_pen']}  (expected unserved return weight)")
print(f"  N_veh   : {joint['N_veh']}  (active vehicles selected by MILP)")
print()

fwd_total = fwd_kpi["routing_cost_R$"].sum()
rev_total = rev_kpi["routing_cost_R$"].sum()
combined  = fwd_total + rev_total
print("=== Combined Logistics Cost ===")
print(f"  Forward (OR-Tools) : R${fwd_total:.0f}")
print(f"  Reverse (OR-Tools) : R${rev_total:.0f}")
print(f"  Combined           : R${combined:.0f}")
print(f"  Forward share      : {fwd_total/combined*100:.1f}%")
print(f"  Reverse share      : {rev_total/combined*100:.1f}%")

---
## 4. Day 4 EOD Checklist

| Checkpoint | Status | Detail |
|-----------|--------|--------|
| XGBoost return classifier trained | ✅ | ROC-AUC=0.897, PR-AUC=0.472 |
| `data/master_df_v3.parquet` generated | ✅ | 19,207 rows + `return_prob` + `return_flag` |
| `outputs/return_clf_v1.pkl` saved | ✅ | XGBClassifier + Platt calibration |
| 593 orders flagged for reverse pickup | ✅ | 3.1% at threshold 0.30 |
| Reverse VRP — all 11 zones solved | ✅ | 11/11, 946.4 km, R$2,170, 15 vehicles |
| `outputs/reverse_routes.json` saved | ✅ | Full stop-level pickup routes |
| `outputs/reverse_kpi_summary.csv` saved | ✅ | Zone KPIs |
| `joint_optimizer.py` Z computable | ✅ | Status=Optimal, Z=54.38 |
| `outputs/joint_optimizer_result.json` | ✅ | α=β=γ=δ=0.25 |
| `notebooks/04_return_ml.ipynb` | ✅ | 26 cells (team notebook) |
| `notebooks/05_reverse_vrp.ipynb` | ✅ | 16 cells (team notebook) |
| `day4_session_summary.ipynb` | ✅ | This notebook (8 cells) |

---
## 5. Day 5 Task List

Per the roadmap (page 10–11):

### Task 1 — SDVRP Hybrid Prototype (simultaneous fwd+rev, 1 zone)
Build a single OR-Tools solve combining delivery + pickup in one route, enforcing:

```python
# At every stop t: 0 ≤ cumulative_delivered(t) - cumulative_collected(t) ≤ VEHICLE_CAPACITY_G
```

**Target zone:** Zone 8 (largest combined cost — R$250 reverse + R$320 forward = R$570).  
**Expected saving:** 15–25% vs separate fwd + rev.  
**File:** `src/joint_optimizer.py` → `solve_sdvrp_hybrid(zone, return_nodes)`

### Task 2 — Z weight sensitivity analysis
Vary α, β, γ, δ over a grid; plot Z surface; identify dominant cost component.

### Day 5 Outputs Required
| Output | Status |
|--------|--------|
| `src/joint_optimizer.py` — `solve_sdvrp_hybrid()` | To do |
| `outputs/sdvrp_zone8_result.json` | To do |
| `outputs/z_sensitivity.csv` | To do |
| `notebooks/pritam_temp_notebooks/day5_session_summary.ipynb` | To do |

In [ ]:
import os

print("=== Day 4 Output Verification ===\n")

checks = [
    ("data/master_df_v3.parquet",             lambda p: pd.read_parquet(p).shape[0] == 19207),
    ("outputs/return_clf_v1.pkl",             os.path.exists),
    ("outputs/return_classifier_metrics.json",os.path.exists),
    ("outputs/reverse_routes.json",           lambda p: len(json.load(open(p))) == 11),
    ("outputs/reverse_kpi_summary.csv",       os.path.exists),
    ("outputs/joint_optimizer_result.json",   lambda p: json.load(open(p))["status"] == "Optimal"),
]

all_ok = True
for path, check in checks:
    try:
        ok = check(f"../../{path}")
    except Exception:
        ok = False
    if not ok:
        all_ok = False
    print(f"  {'✓' if ok else '✗'}  {path}")

print()
print("All Day 4 deliverables verified ✓" if all_ok else "⚠ Some checks failed")